# 03_preprocess_for_ml

**Objective:** Produce the tree-ready and linear-ready feature tables from the focal sample, mapping countries to regions and preparing categorical and OECD feature columns for modelling.

**Inputs:** `2_final_sample_focus.csv`.

**Outputs:** `3_final_sample_focus_tree_ready.csv`; `3_final_sample_focus_linear_ready.csv`.

## Imports

In [ ]:
# Imports
import pandas as pd
from pathlib import Path

## Configuration: paths and feature/category column lists

In [ ]:
# Configuration: paths and feature/category column lists
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
SRC         = PROC / '2_final_sample_focus.csv'
OUT_TREE    = PROC / '3_final_sample_focus_tree_ready.csv'
OUT_LINEAR  = PROC / '3_final_sample_focus_linear_ready.csv'

In [ ]:
# Country-to-region mapping for European VC-market segments
COUNTRY_TO_REGION = {
    'Germany': 'DACH', 'Austria': 'DACH', 'Switzerland': 'DACH', 'Liechtenstein': 'DACH',
    'United Kingdom': 'UK_Ireland', 'Ireland': 'UK_Ireland',
    'Sweden': 'Nordic', 'Denmark': 'Nordic', 'Norway': 'Nordic',
    'Finland': 'Nordic', 'Iceland': 'Nordic',
    'The Netherlands': 'Benelux', 'Belgium': 'Benelux', 'Luxembourg': 'Benelux',
    'France': 'France',
    'Italy': 'Southern', 'Spain': 'Southern', 'Portugal': 'Southern', 'Greece': 'Southern',
    'Poland': 'Eastern', 'Russian Federation': 'Eastern', 'Czech Republic': 'Eastern',
    'Slovakia (Slovak Republic)': 'Eastern', 'Latvia': 'Eastern',
    'Bulgaria': 'Eastern', 'Hungary': 'Eastern', 'Lithuania': 'Eastern',
    'Romania': 'Eastern', 'Slovenia': 'Eastern',
}

In [ ]:
# OECD-derived patent-quality feature columns that may carry NaN values
OECD_FEATURE_COLS = [
    'bwd_cits_mean',    'bwd_cits_max',
    'npl_cits_mean',    'npl_cits_max',
    'originality_mean', 'originality_max',
    'radicalness_mean', 'radicalness_max',
    'claims_mean',      'claims_max',
    'family_size_mean', 'family_size_max',
    'patent_scope_mean','patent_scope_max',
]

CATEGORICAL_FEATURES = ['country_region', 'tech_field_group']

## Main pipeline: region mapping, then tree-ready and linear-ready file construction

In [ ]:
# Main pipeline: map regions, then build tree-ready and linear-ready files
def main():
    df = pd.read_csv(SRC)

    df['country_region'] = df['country'].map(COUNTRY_TO_REGION).fillna('Other')
    n_other = int((df['country_region'] == 'Other').sum())
    if n_other:
        unmapped = df.loc[df['country_region'] == 'Other', 'country'].unique().tolist()
        print(f'WARNING: {n_other} rows unmapped to a region: {unmapped}')

    df = df.drop(columns=['country'])
    print(f'Source: {len(df):,} rows  |  country -> 7 regions')
    print(df['country_region'].value_counts().to_string())
    print()

    df_tree = df.copy()
    for c in OECD_FEATURE_COLS:
        df_tree[f'{c}_missing'] = df_tree[c].isna().astype(int)
        df_tree[c] = df_tree[c].fillna(0)

    df_tree.to_csv(OUT_TREE, index=False)
    print(f'Wrote {OUT_TREE.name}  ({df_tree.shape[0]:,} rows x {df_tree.shape[1]} cols)')
    print(f'  +{len(OECD_FEATURE_COLS)} missingness-indicator columns; categoricals kept as strings')

    df_lin = df.copy()

    tech_label = df_lin['tech_field_group'].copy()
    df_lin = pd.get_dummies(
        df_lin, columns=CATEGORICAL_FEATURES, drop_first=True, dtype=int,
    )
    df_lin['tech_field_group'] = tech_label.values

    nan_counts = {c: int(df_lin[c].isna().sum())
                  for c in OECD_FEATURE_COLS if df_lin[c].isna().any()}
    df_lin.to_csv(OUT_LINEAR, index=False)
    print(f'\nWrote {OUT_LINEAR.name}  ({df_lin.shape[0]:,} rows x {df_lin.shape[1]} cols)')
    print('  OECD features left raw (NaN preserved); winsor + impute now fold-internal in 11.')
    if nan_counts:
        print('  NaN counts (to be imputed within train folds):', nan_counts)

    print('\nReminders for the training script:')
    print('  * Apply excluded_startup == 0 filter at train time (or include for sensitivity).')
    print('  * Standardise continuous features within each train fold, not on full data.')
    print('  * For stage outcomes, decide whether to also drop the 479 no-equity-ladder rows.')

## Run when executed as a script

In [ ]:
# Entry point
if __name__ == '__main__':
    main()